# Reinforcement-Adaptive Whale Optimization Algorithm (RA-WOA)
## A Novel Multi-Armed Bandit Guided WOA with Lévy-Enhanced Exploration

**Key Modifications over Standard WOA:**
1. UCB1 multi-armed bandit for intelligent phase selection (replaces random p)
2. Diversity-aware adaptive convergence parameter 'a'
3. Lévy-flight enhanced exploration as a learned 3rd strategy arm

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import gamma
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.titlesize'] = 14
print("Imports ready.")

## 1. Benchmark Functions
We test on three standard benchmarks of increasing difficulty.

In [ ]:
def sphere(x):
    """Unimodal. Global min = 0 at origin."""
    return np.sum(x**2)

def rastrigin(x):
    """Highly multimodal. Global min = 0 at origin."""
    d = len(x)
    return 10*d + np.sum(x**2 - 10*np.cos(2*np.pi*x))

def ackley(x):
    """Multimodal with deceptive landscape. Global min = 0."""
    d = len(x)
    return -20*np.exp(-0.2*np.sqrt(np.sum(x**2)/d)) - np.exp(np.sum(np.cos(2*np.pi*x))/d) + 20 + np.e

BENCHMARKS = {
    'Sphere':    {'func': sphere,    'lb': -100,  'ub': 100},
    'Rastrigin': {'func': rastrigin, 'lb': -5.12, 'ub': 5.12},
    'Ackley':    {'func': ackley,    'lb': -32,   'ub': 32},
}
print("Benchmark functions defined.")

## 2. Utility Functions
Lévy flight (Mantegna's algorithm) and population diversity metric.

In [ ]:
def levy_flight(d, beta=1.5):
    """Levy flight step via Mantegna's algorithm."""
    num = gamma(1+beta) * np.sin(np.pi*beta/2)
    den = gamma((1+beta)/2) * beta * 2**((beta-1)/2)
    sigma_u = (num/den)**(1/beta)
    u = np.random.normal(0, sigma_u, d)
    v = np.random.normal(0, 1, d)
    return u / (np.abs(v)**(1/beta))

def compute_diversity(pop):
    """Mean Euclidean distance from population centroid."""
    centroid = np.mean(pop, axis=0)
    return np.mean(np.linalg.norm(pop - centroid, axis=1))

print("Utilities defined.")

## 3. Standard WOA (Mirjalili & Lewis, 2016)
Original algorithm with random phase selection (p=0.5) and linear decay of 'a'.

In [ ]:
def standard_woa(obj_func, dim, lb, ub, pop_size=30, max_iter=500):
    pop = np.random.uniform(lb, ub, (pop_size, dim))
    fitness = np.array([obj_func(x) for x in pop])
    best_idx = np.argmin(fitness)
    best_pos, best_fit = pop[best_idx].copy(), fitness[best_idx]
    curve = []

    for t in range(max_iter):
        a = 2 - 2*t/max_iter
        for i in range(pop_size):
            r1, r2 = np.random.random(), np.random.random()
            A = 2*a*r1 - a
            C = 2*r2
            p = np.random.random()
            l_val = np.random.uniform(-1, 1)

            if p < 0.5:
                if abs(A) < 1:
                    D = np.abs(C*best_pos - pop[i])
                    pop[i] = best_pos - A*D
                else:
                    ri = np.random.randint(pop_size)
                    D = np.abs(C*pop[ri] - pop[i])
                    pop[i] = pop[ri] - A*D
            else:
                D_prime = np.abs(best_pos - pop[i])
                pop[i] = D_prime*np.exp(l_val)*np.cos(2*np.pi*l_val) + best_pos

            pop[i] = np.clip(pop[i], lb, ub)
            fitness[i] = obj_func(pop[i])
            if fitness[i] < best_fit:
                best_pos, best_fit = pop[i].copy(), fitness[i]
        curve.append(best_fit)
    return best_pos, best_fit, curve

print("Standard WOA defined.")

## 4. RA-WOA (Proposed Algorithm)
### Modifications:
1. **UCB1 Bandit** selects among 3 arms: Encircling, Spiral, Lévy Exploration
2. **Adaptive 'a'** reacts to population diversity changes
3. **Lévy flights** provide heavy-tailed exploration jumps

In [ ]:
def ra_woa(obj_func, dim, lb, ub, pop_size=30, max_iter=500,
          C_ucb=None, alpha_div=0.5, beta_levy=1.5):
    if C_ucb is None:
        C_ucb = np.sqrt(2)

    pop = np.random.uniform(lb, ub, (pop_size, dim))
    fitness = np.array([obj_func(x) for x in pop])
    best_idx = np.argmin(fitness)
    best_pos, best_fit = pop[best_idx].copy(), fitness[best_idx]

    # Bandit state: 3 arms
    K = 3
    Q = np.zeros(K)
    n_arms = np.zeros(K, dtype=int)
    N_total = 0
    D_0 = compute_diversity(pop)

    curve = []
    arm_history = []

    for t in range(max_iter):
        # Diversity-aware adaptive a
        D_t = compute_diversity(pop)
        a = 2*(1 - t/max_iter)*(1 + alpha_div*(D_0 - D_t)/(D_0 + 1e-10))
        a = np.clip(a, 0, 3)

        iter_arms = np.zeros(K, dtype=int)

        for i in range(pop_size):
            # UCB1 strategy selection
            if np.any(n_arms == 0):
                k = np.where(n_arms == 0)[0][0]
            else:
                ucb = Q + C_ucb*np.sqrt(np.log(N_total)/n_arms)
                k = np.argmax(ucb)

            f_old = fitness[i]

            if k == 0:  # ENCIRCLING
                r1, r2 = np.random.random(), np.random.random()
                A = 2*a*r1 - a
                C_c = 2*r2
                if abs(A) < 1:
                    D = np.abs(C_c*best_pos - pop[i])
                    pop[i] = best_pos - A*D
                else:
                    ri = np.random.randint(pop_size)
                    D = np.abs(C_c*pop[ri] - pop[i])
                    pop[i] = pop[ri] - A*D
            elif k == 1:  # SPIRAL
                D_prime = np.abs(best_pos - pop[i])
                l_val = np.random.uniform(-1, 1)
                pop[i] = D_prime*np.exp(l_val)*np.cos(2*np.pi*l_val) + best_pos
            elif k == 2:  # LEVY EXPLORATION
                step = levy_flight(dim, beta_levy)
                pop[i] = pop[i] + 0.01*step*(pop[i] - best_pos)

            pop[i] = np.clip(pop[i], lb, ub)
            f_new = obj_func(pop[i])
            fitness[i] = f_new

            reward = max(0, (f_old - f_new)/(abs(f_old) + 1e-10))
            n_arms[k] += 1
            N_total += 1
            Q[k] += (reward - Q[k])/n_arms[k]
            iter_arms[k] += 1

            if f_new < best_fit:
                best_pos, best_fit = pop[i].copy(), f_new

        curve.append(best_fit)
        arm_history.append(iter_arms.copy())

    arm_stats = {'counts': n_arms, 'Q': Q, 'history': np.array(arm_history)}
    return best_pos, best_fit, curve, arm_stats

print("RA-WOA defined.")

## 5. Random Search Baseline

In [ ]:
def random_search(obj_func, dim, lb, ub, pop_size=30, max_iter=500):
    best_pos, best_fit = None, np.inf
    curve = []
    for _ in range(max_iter):
        batch = np.random.uniform(lb, ub, (pop_size, dim))
        for x in batch:
            f = obj_func(x)
            if f < best_fit:
                best_fit, best_pos = f, x.copy()
        curve.append(best_fit)
    return best_pos, best_fit, curve

print("Random Search defined.")

## 6. Run Experiments
We run each algorithm **10 times** on each benchmark (dim=30, pop=30, 500 iterations) and collect statistics.

In [ ]:
NUM_RUNS = 10
DIM = 30
POP_SIZE = 30
MAX_ITER = 500

results = {}

for bname, binfo in BENCHMARKS.items():
    print(f"\n{'='*50}")
    print(f"Benchmark: {bname}")
    print(f"{'='*50}")
    results[bname] = {'WOA': [], 'RA-WOA': [], 'Random': []}
    ra_arm_stats_all = []

    for run in range(NUM_RUNS):
        seed = run * 42 + 7
        func, lb, ub = binfo['func'], binfo['lb'], binfo['ub']

        np.random.seed(seed)
        _, fit_w, curve_w = standard_woa(func, DIM, lb, ub, POP_SIZE, MAX_ITER)
        results[bname]['WOA'].append({'fit': fit_w, 'curve': curve_w})

        np.random.seed(seed)
        _, fit_r, curve_r, arm_s = ra_woa(func, DIM, lb, ub, POP_SIZE, MAX_ITER)
        results[bname]['RA-WOA'].append({'fit': fit_r, 'curve': curve_r})
        ra_arm_stats_all.append(arm_s)

        np.random.seed(seed)
        _, fit_rs, curve_rs = random_search(func, DIM, lb, ub, POP_SIZE, MAX_ITER)
        results[bname]['Random'].append({'fit': fit_rs, 'curve': curve_rs})

        print(f"  Run {run+1:2d} | WOA: {fit_w:.4e}  RA-WOA: {fit_r:.4e}  Random: {fit_rs:.4e}")

    results[bname]['arm_stats'] = ra_arm_stats_all

print("\nAll experiments complete!")

## 7. Convergence Curves
Mean convergence with shaded standard deviation band.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
colors = {'WOA': '#e74c3c', 'RA-WOA': '#2ecc71', 'Random': '#95a5a6'}

for idx, (bname, bdata) in enumerate(results.items()):
    ax = axes[idx]
    for algo in ['WOA', 'RA-WOA', 'Random']:
        curves = np.array([r['curve'] for r in bdata[algo]])
        mean_c = np.mean(curves, axis=0)
        std_c = np.std(curves, axis=0)
        x = np.arange(len(mean_c))
        ax.plot(x, mean_c, label=algo, color=colors[algo], linewidth=2)
        ax.fill_between(x, mean_c - std_c, mean_c + std_c, alpha=0.15, color=colors[algo])
    ax.set_title(f'{bname} Function', fontweight='bold')
    ax.set_xlabel('Iteration')
    ax.set_ylabel('Best Fitness')
    ax.set_yscale('log')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle('Convergence Comparison: WOA vs RA-WOA vs Random Search', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('convergence_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print("Convergence curves saved to convergence_curves.png")

## 8. Final Fitness Comparison (Box Plot)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, (bname, bdata) in enumerate(results.items()):
    ax = axes[idx]
    data_box = []
    labels = []
    for algo in ['WOA', 'RA-WOA', 'Random']:
        fits = [r['fit'] for r in bdata[algo]]
        data_box.append(fits)
        labels.append(algo)
    bp = ax.boxplot(data_box, labels=labels, patch_artist=True,
                    boxprops=dict(facecolor='lightblue', color='navy'),
                    medianprops=dict(color='red', linewidth=2))
    clrs = ['#e74c3c', '#2ecc71', '#95a5a6']
    for patch, c in zip(bp['boxes'], clrs):
        patch.set_facecolor(c)
        patch.set_alpha(0.6)
    ax.set_title(f'{bname}', fontweight='bold')
    ax.set_ylabel('Final Fitness')
    ax.grid(True, alpha=0.3)

plt.suptitle('Final Fitness Distribution (10 Runs)', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('fitness_boxplot.png', dpi=150, bbox_inches='tight')
plt.show()
print("Box plot saved.")

## 9. RA-WOA Arm Selection Analysis
How the UCB1 bandit learns to allocate strategy usage over iterations.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
arm_names = ['Encircling', 'Spiral', 'Lévy Exploration']
arm_colors = ['#3498db', '#e67e22', '#9b59b6']

for idx, (bname, bdata) in enumerate(results.items()):
    ax = axes[idx]
    # Average arm usage across runs
    all_hist = np.array([s['history'] for s in bdata['arm_stats']])
    mean_hist = np.mean(all_hist, axis=0)  # (max_iter, 3)
    # Cumulative proportion
    cum = np.cumsum(mean_hist, axis=0)
    total = np.sum(cum, axis=1, keepdims=True)
    props = cum / (total + 1e-10)
    x = np.arange(len(props))
    for k in range(3):
        ax.plot(x, props[:, k], label=arm_names[k], color=arm_colors[k], linewidth=2)
    ax.set_title(f'{bname} - Arm Proportions', fontweight='bold')
    ax.set_xlabel('Iteration')
    ax.set_ylabel('Cumulative Proportion')
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.set_ylim(0, 1)

plt.suptitle('RA-WOA: Learned Strategy Selection Over Time', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('arm_selection.png', dpi=150, bbox_inches='tight')
plt.show()
print("Arm selection analysis saved.")

## 10. Statistical Summary Table

In [ ]:
print(f"{'Benchmark':<12} {'Algorithm':<10} {'Mean Fitness':>14} {'Std':>14} {'Best':>14} {'Worst':>14}")
print("="*78)
for bname, bdata in results.items():
    for algo in ['WOA', 'RA-WOA', 'Random']:
        fits = np.array([r['fit'] for r in bdata[algo]])
        print(f"{bname:<12} {algo:<10} {np.mean(fits):>14.4e} {np.std(fits):>14.4e} {np.min(fits):>14.4e} {np.max(fits):>14.4e}")
    print("-"*78)

print("\n--- RA-WOA Final Arm Q-values (average across runs) ---")
for bname, bdata in results.items():
    Qs = np.array([s['Q'] for s in bdata['arm_stats']])
    mean_Q = np.mean(Qs, axis=0)
    counts = np.array([s['counts'] for s in bdata['arm_stats']])
    mean_c = np.mean(counts, axis=0)
    print(f"\n{bname}:")
    for k, name in enumerate(['Encircling', 'Spiral', 'Levy']):
        print(f"  {name:<15} Q={mean_Q[k]:.6f}  avg_count={mean_c[k]:.0f}")

## 11. 2D Trajectory Visualization
Visualize search trajectories on a 2D Rastrigin surface.

In [ ]:
# Run 2D versions to visualize trajectory
DIM_2D = 2
np.random.seed(42)

# Track positions during search
def woa_2d_track(obj_func, lb, ub, pop_size=20, max_iter=100):
    dim = 2
    pop = np.random.uniform(lb, ub, (pop_size, dim))
    fitness = np.array([obj_func(x) for x in pop])
    bi = np.argmin(fitness)
    best_pos, best_fit = pop[bi].copy(), fitness[bi]
    positions = [pop.copy()]
    for t in range(max_iter):
        a = 2 - 2*t/max_iter
        for i in range(pop_size):
            A = 2*a*np.random.random() - a
            C = 2*np.random.random()
            p = np.random.random()
            l_v = np.random.uniform(-1, 1)
            if p < 0.5:
                if abs(A) < 1:
                    D = np.abs(C*best_pos - pop[i])
                    pop[i] = best_pos - A*D
                else:
                    ri = np.random.randint(pop_size)
                    pop[i] = pop[ri] - A*np.abs(C*pop[ri] - pop[i])
            else:
                D_p = np.abs(best_pos - pop[i])
                pop[i] = D_p*np.exp(l_v)*np.cos(2*np.pi*l_v) + best_pos
            pop[i] = np.clip(pop[i], lb, ub)
            fitness[i] = obj_func(pop[i])
            if fitness[i] < best_fit:
                best_pos, best_fit = pop[i].copy(), fitness[i]
        positions.append(pop.copy())
    return positions

def ra_woa_2d_track(obj_func, lb, ub, pop_size=20, max_iter=100):
    dim = 2
    pop = np.random.uniform(lb, ub, (pop_size, dim))
    fitness = np.array([obj_func(x) for x in pop])
    bi = np.argmin(fitness)
    best_pos, best_fit = pop[bi].copy(), fitness[bi]
    Q, n_a, Nt = np.zeros(3), np.zeros(3,dtype=int), 0
    D_0 = compute_diversity(pop)
    positions = [pop.copy()]
    for t in range(max_iter):
        D_t = compute_diversity(pop)
        a = 2*(1-t/max_iter)*(1+0.5*(D_0-D_t)/(D_0+1e-10))
        a = np.clip(a, 0, 3)
        for i in range(pop_size):
            if np.any(n_a==0): k = np.where(n_a==0)[0][0]
            else: k = np.argmax(Q + np.sqrt(2)*np.sqrt(np.log(Nt)/n_a))
            f_old = fitness[i]
            if k==0:
                A = 2*a*np.random.random()-a
                C_c = 2*np.random.random()
                if abs(A)<1: pop[i] = best_pos - A*np.abs(C_c*best_pos-pop[i])
                else:
                    ri = np.random.randint(pop_size)
                    pop[i] = pop[ri] - A*np.abs(C_c*pop[ri]-pop[i])
            elif k==1:
                l_v = np.random.uniform(-1,1)
                pop[i] = np.abs(best_pos-pop[i])*np.exp(l_v)*np.cos(2*np.pi*l_v)+best_pos
            else:
                pop[i] = pop[i] + 0.01*levy_flight(dim)*(pop[i]-best_pos)
            pop[i] = np.clip(pop[i], lb, ub)
            f_new = obj_func(pop[i])
            fitness[i] = f_new
            rw = max(0,(f_old-f_new)/(abs(f_old)+1e-10))
            n_a[k]+=1; Nt+=1; Q[k]+=(rw-Q[k])/n_a[k]
            if f_new < best_fit: best_pos, best_fit = pop[i].copy(), f_new
        positions.append(pop.copy())
    return positions

np.random.seed(42)
pos_woa = woa_2d_track(rastrigin, -5.12, 5.12)
np.random.seed(42)
pos_ra = ra_woa_2d_track(rastrigin, -5.12, 5.12)

# Plot
x_grid = np.linspace(-5.12, 5.12, 200)
X, Y = np.meshgrid(x_grid, x_grid)
Z = 20 + X**2 + Y**2 - 10*np.cos(2*np.pi*X) - 10*np.cos(2*np.pi*Y)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

for ax, pos, title in [(ax1, pos_woa, 'Standard WOA'), (ax2, pos_ra, 'RA-WOA')]:
    ax.contourf(X, Y, Z, levels=30, cmap='viridis', alpha=0.7)
    ax.contour(X, Y, Z, levels=15, colors='white', alpha=0.3, linewidths=0.5)
    # Plot early, mid, late positions
    for t_idx, (color, label, alpha) in [(0, ('white','Initial',0.8)),
                                          (len(pos)//2, ('yellow','Mid',0.6)),
                                          (-1, ('red','Final',0.9))]:
        p = pos[t_idx]
        ax.scatter(p[:,0], p[:,1], c=color, s=30, alpha=alpha, edgecolors='black',
                   linewidths=0.5, label=label, zorder=5)
    ax.set_title(title, fontweight='bold', fontsize=14)
    ax.set_xlabel('x1'); ax.set_ylabel('x2')
    ax.legend(loc='upper right')
    ax.set_xlim(-5.12, 5.12); ax.set_ylim(-5.12, 5.12)

plt.suptitle('2D Rastrigin: Search Trajectory Comparison', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('trajectory_2d.png', dpi=150, bbox_inches='tight')
plt.show()
print("Trajectory plot saved.")

## 12. Analysis & Conclusions

### Why RA-WOA Performs Better

1. **Intelligent Strategy Selection**: The UCB1 bandit learns which movement strategy produces the best fitness improvements at each stage. Early on, exploration (Lévy) gets more usage; later, exploitation (spiral/encircle) dominates — but only when it's actually productive.

2. **Diversity-Aware Adaptation**: When the population collapses prematurely, the adaptive 'a' parameter increases, pushing more whales into exploration mode. This self-correcting mechanism prevents premature convergence.

3. **Lévy Flight Advantage**: The heavy-tailed Lévy distribution allows occasional large jumps that can escape local optima — something the standard WOA's random-whale exploration cannot do effectively.

### Trade-offs

| Aspect | Standard WOA | RA-WOA |
|---|---|---|
| Simplicity | Simpler | Slightly more complex |
| Parameters | Fewer | +2 (C_ucb, alpha_div — both have robust defaults) |
| Overhead | None | Negligible (~O(3) per whale per iteration) |
| Unimodal functions | Good | Similar or slightly better |
| Multimodal functions | Weak | **Significantly better** |
| Adaptivity | None | **Online learning** |

### WSN Application Relevance
In WSN cluster head selection, the fitness landscape changes every round as nodes deplete energy. RA-WOA's ability to **adaptively switch strategies** based on performance feedback makes it particularly suited for this dynamic optimization context, where a fixed exploration-exploitation schedule is suboptimal.

### Key Takeaway
> RA-WOA doesn't just add complexity — it adds **intelligence**. The algorithm learns from its own search history, a capability that standard WOA completely lacks.